# Canadian principal crops data analysis (R/SQLite)

## Introduction

This notebook contains solutions to the tasks given by IBM Data Analytics with Excel and R course programme. For an actual project building on the question and dataset, see the [final project notebook](https://github.com/tros01/ibm_data_analyst/blob/main/r_m6_annual_crop_data-final_project.ipynb).

### Scenario

We have been hired as a data analyst by a US venture capital company, which has recently invested into the microbrewery and microdistillery industry. Our task is to perform a high-level analysis of crop production in Canada in order to inform our employer’s supply chain decisions. We therefore need to help the stakeholders understand the supply and price volatility of select crop types.

We are given [Canadian principal crops data](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMRP0203ENSkillsNetwork23863830-2021-01-01&pid=3210035901) (1908-2020) listing the measures of pricipal crop produce subsettable by province or territory, [farm product prices](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMRP0203ENSkillsNetwork23863830-2021-01-01&pid=3210007701) (1980-2020) listing the average prices of farm produce and [Bank of Canada daily average exchange rates](https://www.bankofcanada.ca/rates/exchange/daily-exchange-rates?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMRP0203ENSkillsNetwork23863830-2021-01-01).

In [81]:
# Libraries
library(tidyverse)
library(RSQLite)
library(DBI)

## Workings

### Helpers

In [166]:
# SQL tab constructor
construct_tab <- function(channel, construction_query) {
    c <- channel
    q <- construction_query

    # Execute channel-query
    df <- tryCatch(
        dbExecute(c, q),
        error = function(e) e
    )

    # Check whether df is an error object
    if (inherits(df, "error")) {
        cat("An error has occurred.\n")
        print(df$message)
    } else {
        cat("Table has been created successfully.\n")
    }
}

# Set up Jupyter graph display
options(
  repr.plot.width  = 10,
  repr.plot.height = 6,
  repr.plot.res    = 150
)

### Construct the database and load up the data

In [83]:
# Annual Crop Data
acd_path <- r"(https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Annual_Crop_Data.csv)"

# Farm product prices
fpp_path <- r"(https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Monthly_Farm_Prices.csv)"

# Daily FX Data
dfx_path <- r"(https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Daily_FX.csv)"

# Monthly FX Data
mfx_path <- r"(https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-RP0203EN-SkillsNetwork/labs/Final%20Project/Monthly_FX.csv)"

In [84]:
# Establish an RSQLite connection
c <- dbConnect(SQLite(), "data/FinalDB_final_project.sqlite")

In [85]:
# Drop the tabs if they already exist
tab_vec <- c("crop_data", "prod_prices", "daily_fx", "monthly_fx")

for (tab in tab_vec) {
    q <- sprintf("
    DROP TABLE IF EXISTS %s;
    ", tab)
    dbExecute(c, q)
}

In [86]:
# Load the csvs into a list of data frames
path_vec <- c(acd_path, fpp_path, dfx_path, mfx_path)
name_vec <- c("acd_path", "fpp_path", "dfx_path", "mfx_path")
path_dic <- setNames(path_vec, name_vec)

data_list <- list()

for (path_name in names(path_dic)) {
    # Load up the dataset and clean the column names
    ds <- read_csv(
        path_dic[[path_name]],
        show_col_types = FALSE
    ) |> janitor::clean_names()
    
    # Add the df into the list
    data_list[[path_name]] <- ds

    # Obtain the column names for tab construction
    cat(path_name, "columns:\n", colnames(data_list[[path_name]]), "\n\n")
    # Obtain the data types
    #cat("Structure":\n", str(data_list[[path_name]]), "\n\n")
}

acd_path columns:
 cd_id year crop_type geo seeded_area harvested_area production avg_yield 

fpp_path columns:
 cd_id date crop_type geo price_prermt 

dfx_path columns:
 dfx_id date fxusdcad 

mfx_path columns:
 dfx_id date fxusdcad 



In [87]:
# Construct the tabs
construction_queries <- c(
    "
    CREATE TABLE IF NOT EXISTS crop_data (
        cd_id INTEGER NOT NULL,
        year DATE NOT NULL,
        crop_type VARCHAR(20) NOT NULL,
        geo VARCHAR(20) NOT NULL,
        seeded_area INTEGER NOT NULL,
        harvested_area INTEGER NOT NULL,
        production INTEGER NOT NULL,
        avg_yield INTEGER NOT NULL,
        PRIMARY KEY(cd_id)
    );
    ",
    "
    CREATE TABLE IF NOT EXISTS prod_prices (
        cd_id INTEGER NOT NULL,
        date DATE NOT NULL,
        crop_type VARCHAR(20) NOT NULL,
        geo VARCHAR(20) NOT NULL,
        price_prermt INTEGER NOT NULL,
        PRIMARY KEY (cd_id)
    );
    ",
    "
    CREATE TABLE IF NOT EXISTS daily_fx (
        dfx_id INTEGER NOT NULL,
        date DATE NOT NULL,
        fxusdcad FLOAT(6) NOT NULL,
        PRIMARY KEY(dfx_id)
    );
    ",
    "
    CREATE TABLE IF NOT EXISTS montly_fx (
        dfx_id INTEGER NOT NULL,
        date DATE NOT NULL,
        fxusdcad FLOAT(6) NOT NULL,
        PRIMARY KEY(dfx_id)
    );
    "
)
construction_dic <- setNames(construction_queries, tab_vec)

for (tab in names(construction_dic)) {
    q <- construction_dic[[tab]]
    construct_tab(c, q)
}

Table has been created successfully.
Table has been created successfully.
Table has been created successfully.
Table has been created successfully.


In [88]:
# List tabs
dbListTables(c)

[1] "crop_data"   "daily_fx"    "montly_fx"   "prod_prices"

In [ ]:
# Load up SQLite database
for (i in seq_along(data_list)) {
    # Tab name
    tab_name <- tab_vec[i]
    # Write the tab into the database
    dbWriteTable(c, tab_name, data_list[[i]], overwrite = FALSE, append = TRUE, header = TRUE)
    # Retrieve the row count
    row_count <- dbGetQuery(c, sprintf("SELECT COUNT(*) FROM %s", tab_name)) |> pull()
    
    cat("Rows in", tab_name, ":", row_count, "\n")
}

Rows in  crop_data : 672 
Rows in  prod_prices : 2678 
Rows in  daily_fx : 1033 
Rows in  monthly_fx : 50 


In [90]:
# Clear the memory
rm(data_list)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1642780,87.8,3033185,162,3033185,162.0
Vcells,2952753,22.6,8388608,64,5876417,44.9


### DB queries

In [ ]:
# Format the dates
for (tab in tab_vec) {
    fields <- dbListFields(c, tab)

    if ("year" %in% fields) {
        date_col <- "year"
    } else {
        date_col <- "date"
    }

    # Rename the original date column
    q <- sprintf("
    ALTER TABLE %s RENAME COLUMN %s TO days_int;
    ", tab, date_col)
    dbExecute(c, q)

    # Add a column for date strings
    q <- sprintf("
    ALTER TABLE %s ADD COLUMN days_str TEXT;
    ", tab)
    dbExecute(c, q)

    # Add a column for the formatted date
    q <- sprintf("
    ALTER TABLE %s ADD COLUMN %s TEXT;
    ", tab, date_col)
    dbExecute(c, q)

    # Convert the number into a date-ready string
    # `||` SQL string concatenation operator; msql: CONCAT('hello', ' world')
    q <- sprintf("
    UPDATE %s
    SET days_str = CAST(days_int AS INTEGER) || ' days';
    ", tab)
    dbExecute(c, q)

    # Construct the date
    q <- sprintf("
    UPDATE %s
    SET %s = DATE('1970-01-01', days_str);
    ", tab, date_col)
    dbExecute(c, q)

    # Drop the helper cols
    q <- sprintf("
    ALTER TABLE %s
    DROP days_int;
    ", tab)
    dbExecute(c, q)

    q <- sprintf("
    ALTER TABLE %s
    DROP days_str;
    ", tab)
    dbExecute(c, q)
}

In [92]:
# How many records are in the farm prices dataset?
q <- "
SELECT COUNT(*) FROM prod_prices;
"
a <- dbGetQuery(c, q) |> pull()
cat("There are", a, "records in the prices table.")

There are 2678 records in the prices table.

In [93]:
# Which geographies are included in the farm prices dataset?
q <- "
SELECT DISTINCT(geo) FROM prod_prices;
"
dbGetQuery(c, q)

geo
<chr>
Alberta
Saskatchewan


Our dataset features prices for two unique geographies alone.

In [101]:
# How many hectares of Rye were harvested in Canada in 1968?
q <- "
SELECT harvested_area FROM crop_data
WHERE 
    LOWER(crop_type) = 'rye' AND 
    STRFTIME('%Y', year) = '1968' AND
    geo = 'Canada';
"
a <- dbGetQuery(c, q) |> pull()
cat(a, "hectares of rye were harvested in all of Canada in 1968.")

274100 hectares of rye were harvested in all of Canada in 1968.

In [103]:
# Query and display the first 6 rows of the farm prices table for Rye.
q <- "
SELECT * FROM prod_prices
WHERE LOWER(crop_type) IN ('rye')
LIMIT 6;
"
dbGetQuery(c, q)

cd_id,crop_type,geo,price_prermt,date
<dbl>,<chr>,<chr>,<dbl>,<chr>
4,Rye,Alberta,100.77,1985-01-01
5,Rye,Saskatchewan,109.75,1985-01-01
10,Rye,Alberta,95.05,1985-02-01
11,Rye,Saskatchewan,103.46,1985-02-01
16,Rye,Alberta,96.77,1985-03-01
17,Rye,Saskatchewan,106.38,1985-03-01


In [106]:
# Which provinces grew Barley?
q <- "
SELECT DISTINCT(geo) FROM crop_data
WHERE LOWER(crop_type) = 'barley' AND geo NOT IN ('Canada');
"
dbGetQuery(c, q)

geo
<chr>
Alberta
Saskatchewan


Only Alberta and Saskatchewan have a record of growing barley.

In [109]:
# Find the first and last dates for the farm prices data.
q <- "
SELECT MIN(date) AS first_date, MAX(date) AS last_date FROM prod_prices;
"
a <- dbGetQuery(c, q)
cat("The farm prices dataset spans", a[1,1], "and", a[1,2], ".")

The farm prices dataset spans 1985-01-01 and 2020-12-01 .

In [111]:
# Which crops have ever reached a farm price greater than or equal to $350 per metric tonne?
q <- "
SELECT DISTINCT(crop_type) FROM prod_prices
WHERE price_prermt >= 350;
"
dbGetQuery(c, q)

crop_type
<chr>
Canola


Only canola has ever traded at $350/tonne or higher.

In [114]:
# Rank the crop types harvested in Saskatchewan in the year 2000 by their average yield. Which crop performed best?
q <- "
SELECT crop_type, avg_yield FROM crop_data
WHERE 
    STRFTIME('%Y', year) IN ('2000') AND
    geo IN ('Saskatchewan')
ORDER BY avg_yield DESC;
"
dbGetQuery(c, q)

crop_type,avg_yield
<chr>,<dbl>
Barley,2800
Wheat,2200
Rye,2100
Canola,1400


Barley won the contest for the highest mean yield in Saskatchewan in 2000.

In [117]:
# Rank the crops and geographies by their average yield (KG per hectare) since the year 2000. Which crop and province had the highest average yield since the year 2000?
q <- "
SELECT crop_type, geo, avg_yield, year FROM crop_data
WHERE STRFTIME('%Y', year) >= '2000'
ORDER BY avg_yield DESC
LIMIT 10;
"
dbGetQuery(c, q)

crop_type,geo,avg_yield,year
<chr>,<chr>,<dbl>,<chr>
Barley,Alberta,4100,2013-12-31
Barley,Alberta,4100,2016-12-31
Barley,Alberta,3980,2020-12-31
Wheat,Alberta,3900,2013-12-31
Barley,Canada,3900,2016-12-31
Wheat,Alberta,3900,2016-12-31
Barley,Alberta,3900,2017-12-31
Barley,Alberta,3890,2019-12-31
Barley,Canada,3820,2020-12-31


The top three spots for the highest average yield are taken by Albertan barley for their harvests in 2013, 2016 and 2020.

In [147]:
# Use a subquery to determine how much wheat was harvested in Canada in the most recent year of the data.
q <- "
SELECT crop_type, geo, production, year FROM crop_data
WHERE
    geo = 'Canada' AND
    LOWER(crop_type) = 'wheat' AND
    STRFTIME('%Y', year) IN (SELECT STRFTIME('%Y', MAX(year)) FROM crop_data);
"
a <- dbGetQuery(c, q)
cat("Results:\n")
print(a)
cat("\n\n", a[1, "production"], "kg of wheat was harvested in Canada in", str_extract(a[1, "year"], "^\\d+"), ".")

Results:
  crop_type    geo production       year
1     Wheat Canada   35183000 2020-12-31


 35183000 kg of wheat was harvested in Canada in 2020 .

In [159]:
# Use an implicit inner join to calculate the monthly price per metric tonne of Canola grown in Saskatchewan in both Canadian and US dollars. Display the most recent 6 months of the data.
q <- "
SELECT
    p.crop_type, 
    p.geo, 
    p.price_prermt AS price_pmt_cad, 
    (p.price_prermt	/ m.fxusdcad) AS price_pmt_usd,
    p.date
FROM prod_prices AS p, monthly_fx AS m
WHERE
    p.date = m.date AND
    LOWER(p.crop_type) IN ('canola') AND
    p.geo = 'Saskatchewan' AND
    p.date > (SELECT DATE(MAX(date), '-6 months') FROM prod_prices)
ORDER BY p.date DESC;
"
a <- dbGetQuery(c, q)
a

crop_type,geo,price_pmt_cad,price_pmt_usd,date
<chr>,<chr>,<dbl>,<dbl>,<chr>
Canola,Saskatchewan,507.33,396.1128,2020-12-01
Canola,Saskatchewan,495.64,379.2718,2020-11-01
Canola,Saskatchewan,474.80,359.2965,2020-10-01
Canola,Saskatchewan,463.52,350.4057,2020-09-01
Canola,Saskatchewan,464.60,351.3827,2020-08-01
Canola,Saskatchewan,462.88,342.9122,2020-07-01


In [168]:
# Disconnect
dbDisconnect(c)